## 스트리밍 (Streaming)
- LangGraph 애플리케이션을 개발할 때, 실행 중인 그래프의 상태를 실시간으로 모니터링하거나 LLM의 응답을 토큰 단위로 받아보고 싶은 경우,
  
  특히 사용자 경험을 향상시키기 위해서는 긴 작업이 진행되는 동안 중간 결과를 실시간으로 표시하는 것이 중요
  
  $\rightarrow$ LangGraph의 스트리밍(Streaming) 기능은 이러한 요구사항을 해결하기 위해 제공
  
  그래프 실행 과정에서 발생하는 다양한 이벤트와 데이터를 실시간으로 스트리밍할 수 있으며, 여러 모드를 제공하여 원하는 정보만 선택적으로 받을 수 있음

<br>

### 스트리밍의 필요성

<br>

#### 사용자 경험 향상
- 긴 작업 중에도 진행 상황을 실시간으로 보여줄 수 있음
- LLM 응답을 토큰 단위로 스트리밍하여 대기 시간을 줄임
- 사용자는 작업이 진행 중임을 명확히 인지할 수 있음

<br>


#### 디버깅 및 모니터링
- 그래프 실행 과정을 실시간으로 추적할 수 있음
- 각 노드의 상태 변화를 관찰하여 문제를 빠르게 파악할 수 있음
- 프로덕션 환경에서 성능과 동작을 모니터링할 수 있음

<br>

#### 유연한 데이터 처리
- 필요한 데이터만 선택적으로 스트리밍 받을 수 있음
- 다중 에이전트 시스템에서 각 에이전트의 출력을 개별적으로 처리할 수 있음
- 서브그래프의 출력도 포함하여 복잡한 시스템을 효과적으로 관리할 수 있음

<br>

### `LangGraph`의 Stream Mode 종류
- 그래프 전체 상태가 필요하면 `values` 사용
- 상태 변경 사항만 필요하면 `updates` 사용
- LLM 응답을 실시간으로 표시하려면 `messages` 사용
- 진행 상황이나 로그를 전송하려면 `custom` 사용
- 디버깅이 필요하면 `debug` 사용

<br>

<table>
<thead>
<tr>
<th style="text-align: left;">Mode</th>
<th style="text-align: left;">설명</th>
<th style="text-align: left;">주요 용도</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>values</code></td>
<td style="text-align: left;">각 단계 후 전체 상태를 스트리밍</td>
<td style="text-align: left;">그래프 전체 상태 추적</td>
</tr>
<tr>
<td style="text-align: left;"><code>updates</code></td>
<td style="text-align: left;">각 단계의 상태 변화분만 스트리밍</td>
<td style="text-align: left;">효율적인 상태 변경 추적</td>
</tr>
<tr>
<td style="text-align: left;"><code>messages</code></td>
<td style="text-align: left;">LLM 토큰을 (<code>message_chunk</code>, <code>metadata</code>) 튜플로 스트리밍</td>
<td style="text-align: left;">실시간 LLM 응답 표시</td>
</tr>
<tr>
<td style="text-align: left;"><code>custom</code></td>
<td style="text-align: left;">노드 내부에서 사용자 정의 데이터 스트리밍</td>
<td style="text-align: left;">진행 상황, 로그 등 커스텀 정보</td>
</tr>
<tr>
<td style="text-align: left;"><code>debug</code></td>
<td style="text-align: left;">실행 과정의 모든 정보를 상세히 스트리밍</td>
<td style="text-align: left;">디버깅 및 상세 분석</td>
</tr>
</tbody>
</table>

<br>

### 스트리밍 모드 데이터 흐름

<img src='img/1-11-5.png' width=600>

<br>

#### 기본 사용법
- LangGraph의 스트리밍은 `stream()` 메서드를 사용

```python
for chunk in app.stream(
    {"message": "Hello", "count": 0},
    stream_mode="values"  # 원하는 모드 지정
):
    print(chunk)
```

<br>

- 다중 모드를 동시에 지정하면 (`mode`, `chunk`) 튜플 형태로 수신

```python
for mode, chunk in app.stream(
    {"message": "Hello", "count": 0},
    stream_mode=["values", "updates"]
):
    print(f"[{mode}] {chunk}")

```

<br>

<hr>

<br>

### 스트리밍 개요와 `stream_mode`

<br>

#### `stream()` 및 `astream()` 메서드
- LangGraph의 모든 그래프는 동기식 `stream()`과 비동기식 `astream()` 메서드를 제공

In [1]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

In [2]:
class State(TypedDict):
    message: str

In [3]:
graph = StateGraph(State)

In [4]:
def node_func(state: State) -> State:
    return {"message": f"Processed: {state['message']}"}

In [5]:
graph.add_node("process", node_func)
graph.add_edge(START, "process")
graph.add_edge("process", END)

app = graph.compile()

- **동기 스트리밍**

In [6]:
for chunk in app.stream({"message": "Hello"}, stream_mode="values"):
    print(chunk)

{'message': 'Hello'}
{'message': 'Processed: Hello'}


- **비동기 스트리밍**

In [7]:
import asyncio

In [8]:
async def main():
    async for chunk in app.astream({"message": "Hello"}, stream_mode="values"):
        print(chunk)

In [9]:
# asyncio.run(main())

await main()

{'message': 'Hello'}
{'message': 'Processed: Hello'}


<br>

#### 스트리밍 모드 종류

<table>
<thead>
<tr>
<th>모드</th>
<th>설명</th>
<th>출력 형식</th>
</tr>
</thead>
<tbody>
<tr>
<td><code>values</code></td>
<td>각 단계 후 전체 상태</td>
<td>State 딕셔너리</td>
</tr>
<tr>
<td><code>updates</code></td>
<td>상태 변화분만</td>
<td>노드명: 변경값 딕셔너리</td>
</tr>
<tr>
<td><code>messages</code></td>
<td>LLM 토큰 단위</td>
<td>(<code>message_chunk</code>, <code>metadata</code>) 튜플</td>
</tr>
<tr>
<td><code>custom</code></td>
<td>사용자 정의 데이터</td>
<td>자유 형식</td>
</tr>
<tr>
<td><code>debug</code></td>
<td>실행 상세 정보</td>
<td>디버그 이벤트 딕셔너리</td>
</tr>
</tbody>
</table>

<br>

- **`stream()` 호출 시 각 모드별 데이터**

<img src='img/1-11-6.png' width=600>

<br>

#### `values` 모드
- 각 단계 후 그래프의 **전체 상태**를 스트리밍
    - 사용 사례: UI에 전체 상태 표시, 상태 변화 추적
<br>

```python
for state in app.stream(input, stream_mode="values"):
    print(f"전체 상태: {state}")
```

```text
전체 상태: {'message': 'Hello'}
전체 상태: {'message': 'Processed: Hello'}
```

<br>

#### `updates` 모드
- 각 단계의 **상태 변화분**만 스트리밍. 노드 이름과 해당 노드가 반환한 업데이트를 포함
    - 사용 사례: 네트워크 대역폭 절약, 노드별 변경 추적

<br>

```python
for update in app.stream(input, stream_mode="updates"):
    print(f"변경: {update}")
```

```text
변경: {'process': {'message': 'Processed: Hello'}}
```

<br>

#### `messages` 모드
- LLM의 응답을 **토큰 단위**로 실시간 스트리밍
    - 사용 사례: 챗봇 실시간 응답, LLM 출력 모니터링
  
```python
for chunk in app.stream(input, stream_mode="messages"):
    message_chunk, metadata = chunk
    print(message_chunk.content, end="", flush=True)
```

<br>

#### `custom` 모드
- 노드 내부에서 **사용자 정의 데이터**를 자유롭게 스트리밍
  - 사용 사례: 진행 상황 표시, 로그 전송, 중간 결과 알림

```python
from langgraph.config import get_stream_writer

def my_node(state):
    writer = get_stream_writer()
    writer({"progress": 50, "status": "처리 중"})
    return state

for data in app.stream(input, stream_mode="custom"):
    print(f"진행: {data}")
```

<br>

#### `debug` 모드
- 그래프 실행의 **모든 세부 정보**를 스트리밍. 각 이벤트는 `type` 필드를 포함한 딕셔너리로 반환
  - **주요 `type` 값의 의미**
    - **`task`** : 노드 실행 시작 직전, 노드 이름, 입력 상태
    - **`task_result`** : 노드 실행 완료 직후, 노드 이름, 반환된 업데이트
  - 사용 사례: 복잡한 그래프 디버깅, 성능 분석, 문제 추적

```python
for debug_info in app.stream(input, stream_mode="debug"):
    print(f"Debug: {debug_info['type']}")
```

```text
Debug: task
Debug: task_result
```

<br>

#### 다중 모드 동시 사용
- 여러 모드를 리스트로 전달하여 동시에 사용할 시, 각 청크는 (`mode`, `data`) 튜플 형태로 반환

```python
for mode, chunk in app.stream(
    input,
    stream_mode=["values", "updates"]
):
    if mode == "values":
        print(f"전체 상태: {chunk}")
    elif mode == "updates":
        print(f"변경 사항: {chunk}")
```

```text
전체 상태: {'message': 'Hello'}
변경 사항: {'process': {'message': 'Processed: Hello'}}
전체 상태: {'message': 'Processed: Hello'}
```

<br>

#### 활용 에시

```python
# 상태 + 커스텀 데이터 동시 수신
for mode, chunk in app.stream(input, stream_mode=["updates", "custom"]):
    if mode == "updates":
        node_name = list(chunk.keys())[0]
        print(f"[노드 완료] {node_name}")
    elif mode == "custom":
        print(f"[진행 상황] {chunk}")
```

<br>

#### 모드 선택 가이드

<table>
<thead>
<tr>
<th>상황</th>
<th>권장 모드</th>
</tr>
</thead>
<tbody>
<tr>
<td>UI에 전체 상태 표시</td>
<td><code>values</code></td>
</tr>
<tr>
<td>효율적인 변경 추적</td>
<td><code>updates</code></td>
</tr>
<tr>
<td>챗봇 실시간 응답</td>
<td><code>messages</code></td>
</tr>
<tr>
<td>진행률/로그 표시</td>
<td><code>custom</code></td>
</tr>
<tr>
<td>디버깅 및 문제 분석</td>
<td><code>debug</code></td>
</tr>
<tr>
<td>복합 정보 필요</td>
<td>다중 모드 조합</td>
</tr>
</tbody>
</table>

<br>

<hr>

<br>

### 상태 스트리밍 (`values`, `updates`)

<br>

#### `values` 모드
- **전체 상태 스트리밍의 장점**
  - `values` 모드는 각 단계 후 그래프의 전체 상태 스냅샷을 제공

<img src='img/1-11-7.png' width=600>

<br>

- `values` 모드는 각 노드 실행 후 전체 상태 스냅샷을 방출
  - UI 렌더링처럼 항상 최신 전체 상태가 필요한 경우에 적합
  - `values` 모드는 LangGraph가 내부적으로 상태를 병합(merge)한 뒤 전체 스냅샷을 제공
  - 노드 함수가 변경된 키만 반환하더라도 스트림에서는 항상 전체 상태를 받음

In [10]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

In [11]:
class ResearchState(TypedDict):
    topic: str
    sources: list[str]
    summary: str
    confidence: float

In [12]:
graph = StateGraph(ResearchState)

In [14]:
def gather_sources(state: ResearchState) -> dict:
    # 변경된 키만 반환: sources, confidence
    return {
        "sources": state["sources"] + ["Wikipedia", "ArXiv"],
        "confidence": 0.5
    }

def analyze_content(state: ResearchState) -> dict:
    # 변경된 키만 반환: summary, confidence
    return {
        "summary": f"{state['topic']}에 대한 분석 완료",
        "confidence": 0.8
    }

def finalize(state: ResearchState) -> dict:
    # 변경된 키만 반환: summary, confidence
    return {
        "summary": state["summary"] + " (검증됨)",
        "confidence": 1.0
    }

In [15]:
graph.add_node("gather", gather_sources)
graph.add_node("analyze", analyze_content)
graph.add_node("finalize", finalize)

graph.add_edge(START, "gather")
graph.add_edge("gather", "analyze")
graph.add_edge("analyze", "finalize")
graph.add_edge("finalize", END)

app = graph.compile()

In [16]:
initial_state = {
    "topic": "AI 윤리",
    "sources": [],
    "summary": "",
    "confidence": 0.0
}

In [17]:
print("=== values 모드 출력 ===")
for state in app.stream(initial_state, stream_mode="values"):
    print(f"신뢰도: {state['confidence']:.1f}")
    print(f"출처 수: {len(state['sources'])}")
    print(f"요약: {state['summary']}")
    print("-" * 50)

=== values 모드 출력 ===
신뢰도: 0.0
출처 수: 0
요약: 
--------------------------------------------------
신뢰도: 0.5
출처 수: 2
요약: 
--------------------------------------------------
신뢰도: 0.8
출처 수: 2
요약: AI 윤리에 대한 분석 완료
--------------------------------------------------
신뢰도: 1.0
출처 수: 2
요약: AI 윤리에 대한 분석 완료 (검증됨)
--------------------------------------------------


<br>

- **`values` 모드 활용 패턴**
  - **UI 렌더링** : 전체 상태를 받아 UI를 업데이트하기에 이상적

In [ ]:
def render_ui(state: ResearchState):
    """UI를 렌더링하는 함수"""
    bar = "▓" * int(state["confidence"] * 10) + "░" * (10 - int(state["confidence"] * 10))
    sources = ", ".join(state["sources"]) if state["sources"] else "(없음)"
    print(f"  주제: {state['topic']}")
    print(f"  신뢰도: [{bar}] {state['confidence']:.0%}")
    print(f"  출처: {sources}")
    print(f"  요약: {state['summary'] or '(대기 중)'}")
    print()

In [19]:
for state in app.stream(initial_state, stream_mode="values"):
    render_ui(state)

  주제: AI 윤리
  신뢰도: [░░░░░░░░░░] 0%
  출처: (없음)
  요약: (대기 중)

  주제: AI 윤리
  신뢰도: [▓▓▓▓▓░░░░░] 50%
  출처: Wikipedia, ArXiv
  요약: (대기 중)

  주제: AI 윤리
  신뢰도: [▓▓▓▓▓▓▓▓░░] 80%
  출처: Wikipedia, ArXiv
  요약: AI 윤리에 대한 분석 완료

  주제: AI 윤리
  신뢰도: [▓▓▓▓▓▓▓▓▓▓] 100%
  출처: Wikipedia, ArXiv
  요약: AI 윤리에 대한 분석 완료 (검증됨)



<br>

#### `updates` 모드
- **변화분만 추적**
  - `updates` 모드는 각 노드가 실제로 변경한 키만 전송
  - 노드 함수가 변경된 키만 반환하도록 작성하면, `updates` 스트림에서도 해당 키만 수신

<img src='img/1-11-8.png' width=600>

<br>

- **`updates` 모드는 각 노드가 변경한 키만 델타(`delta`)로 방출**
  - 어떤 노드가 무엇을 바꿨는지 정확히 추적할 수 있음

In [ ]:
print("=== updates 모드 출력 ===")
for update in app.stream(initial_state, stream_mode="updates"):
    node_name = list(update.keys())[0]
    changes = update[node_name]

    print(f"노드: {node_name}")
    for key, value in changes.items():
        print(f"  {key}: {value}")
    print("-" * 50)

=== updates 모드 출력 ===
노드: gather
  sources: ['Wikipedia', 'ArXiv']
  confidence: 0.5
--------------------------------------------------
노드: analyze
  summary: AI 윤리에 대한 분석 완료
  confidence: 0.8
--------------------------------------------------
노드: finalize
  summary: AI 윤리에 대한 분석 완료 (검증됨)
  confidence: 1.0
--------------------------------------------------


<br>

- **`updates` 모드 활용 패턴**
  - 변경 사항 로깅

In [21]:
import logging
from datetime import datetime

In [22]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [23]:
for update in app.stream(initial_state, stream_mode="updates"):
    node_name = list(update.keys())[0]
    changes = update[node_name]

    timestamp = datetime.now().strftime("%H:%M:%S")
    logger.info(f"[{timestamp}] 노드 '{node_name}' 실행 완료")

    for key, value in changes.items():
        if key == "confidence":
            logger.info(f"  신뢰도 업데이트: {value:.2f}")
        elif key == "sources" and value:
            logger.info(f"  새 출처 추가: {value}")


INFO:__main__:[16:36:34] 노드 'gather' 실행 완료
INFO:__main__:  새 출처 추가: ['Wikipedia', 'ArXiv']
INFO:__main__:  신뢰도 업데이트: 0.50
INFO:__main__:[16:36:34] 노드 'analyze' 실행 완료
INFO:__main__:  신뢰도 업데이트: 0.80
INFO:__main__:[16:36:34] 노드 'finalize' 실행 완료
INFO:__main__:  신뢰도 업데이트: 1.00


<br>

#### `values` vs `updates` 비교

- **다단계 문서 처리 예시**

In [24]:
from typing_extensions import TypedDict

In [25]:
class DocumentState(TypedDict):
    content: str
    word_count: int
    readability_score: float
    tags: list[str]
    status: str

In [26]:
graph = StateGraph(DocumentState)

- 노드 함수 정의

In [27]:
def preprocess(state: DocumentState) -> dict:
    # 변경된 키만 반환: content, word_count, status
    content = state["content"].strip().lower()
    return {
        "content": content,
        "word_count": len(content.split()),
        "status": "전처리 완료"
    }

def analyze(state: DocumentState) -> dict:
    # 변경된 키만 반환: readability_score, status
    score = min(100, state["word_count"] / 10)
    return {
        "readability_score": score,
        "status": "분석 완료"
    }

def tag_content(state: DocumentState) -> dict:
    # 변경된 키만 반환: tags, status
    tags = ["문서", "분석됨"]
    if state["word_count"] > 100:
        tags.append("긴 문서")
    if state["readability_score"] > 50:
        tags.append("읽기 쉬움")
    return {
        "tags": tags,
        "status": "완료"
    }


In [28]:
graph.add_node("preprocess", preprocess)
graph.add_node("analyze", analyze)
graph.add_node("tag", tag_content)

graph.add_edge(START, "preprocess")
graph.add_edge("preprocess", "analyze")
graph.add_edge("analyze", "tag")
graph.add_edge("tag", END)

app = graph.compile()

- 실행

In [29]:
doc_state = {
    "content": "  LangGraph는 상태 기반 워크플로우를 구축하는 강력한 프레임워크입니다.  ",
    "word_count": 0,
    "readability_score": 0.0,
    "tags": [],
    "status": "초기"
}

In [30]:
print("=== values 모드 ===")
for state in app.stream(doc_state, stream_mode="values"):
    print(f"상태: {state['status']}, 단어 수: {state['word_count']}, 점수: {state['readability_score']:.1f}")

=== values 모드 ===
상태: 초기, 단어 수: 0, 점수: 0.0
상태: 전처리 완료, 단어 수: 7, 점수: 0.0
상태: 분석 완료, 단어 수: 7, 점수: 0.7
상태: 완료, 단어 수: 7, 점수: 0.7


In [31]:
print("\n=== updates 모드 ===")
for update in app.stream(doc_state, stream_mode="updates"):
    node_name = list(update.keys())[0]
    changes = update[node_name]
    print(f"[{node_name}] 변경된 키: {list(changes.keys())}")


=== updates 모드 ===
[preprocess] 변경된 키: ['content', 'word_count', 'status']
[analyze] 변경된 키: ['readability_score', 'status']
[tag] 변경된 키: ['tags', 'status']


<br>

#### 실시간 대시보드 예시
- `values` 모드를 사용하여 전체 상태를 표시

In [35]:
def display_dashboard(state: DocumentState):
    """대시보드 UI 업데이트"""
    tags = ", ".join(state["tags"]) if state["tags"] else "없음"
    print("--- 문서 처리 대시보드 ---")
    print(f"  상태:   {state['status']}")
    print(f"  단어 수: {state['word_count']}")
    print(f"  가독성:  {state['readability_score']:.1f}")
    print(f"  태그:   {tags}")
    print("-" * 26)


In [36]:
for state in app.stream(doc_state, stream_mode="values"):
    display_dashboard(state)

--- 문서 처리 대시보드 ---
  상태:   초기
  단어 수: 0
  가독성:  0.0
  태그:   없음
--------------------------
--- 문서 처리 대시보드 ---
  상태:   전처리 완료
  단어 수: 7
  가독성:  0.0
  태그:   없음
--------------------------
--- 문서 처리 대시보드 ---
  상태:   분석 완료
  단어 수: 7
  가독성:  0.7
  태그:   없음
--------------------------
--- 문서 처리 대시보드 ---
  상태:   완료
  단어 수: 7
  가독성:  0.7
  태그:   문서, 분석됨
--------------------------


<br>

#### 변경 이력 추적 예시
- `updates` 모드를 사용하여 변경 이력을 기록

In [37]:
change_history = []

In [38]:
for update in app.stream(doc_state, stream_mode="updates"):
    node_name = list(update.keys())[0]
    changes = update[node_name]

    change_history.append({
        "node": node_name,
        "timestamp": datetime.now(),
        "changes": changes
    })

In [ ]:
print("\n=== 변경 이력 ===")
for entry in change_history:
    print(f"{entry['timestamp'].strftime('%H:%M:%S')} - {entry['node']}")
    print(f"  변경된 키: {list(entry['changes'].keys())}")


=== 변경 이력 ===
16:55:14 - preprocess
  변경된 키: ['content', 'word_count', 'status']
16:55:14 - analyze
  변경된 키: ['readability_score', 'status']
16:55:14 - tag
  변경된 키: ['tags', 'status']


<br>

<hr>